# Label Propagation Algorithm (LPA) on Cora

Transductive Node Classification on Cora (Planetoid): Iterative diffusion of known labels over graph edges without learnable parameters. This notebook implements the approach with `LabelPropagation`, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `LabelPropagation` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import ops

import k3_node
from k3_node import models as k3_models
from k3_node.datasets import Planetoid

title = "Label Propagation Algorithm (LPA) on Cora"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora")
data = dataset[0]
num_classes = dataset.num_classes

# 2. Label Propagation Model
lpa = k3_models.LabelPropagation(num_layers=10, alpha=0.9)

# 3. Propagate One-Hot Labels
y_one_hot = ops.one_hot(ops.cast(data.y, "int32"), num_classes)
out = lpa(y_one_hot, data.edge_index, mask=data.train_mask)

# 4. Evaluate Test Accuracy
pred = ops.argmax(out, axis=-1)
test_mask = data.test_mask
test_acc = ops.mean(ops.cast(ops.cast(pred[test_mask], "int64") == ops.cast(data.y[test_mask], "int64"), "float32"))
print(f"LPA Accuracy on Unseen Test Nodes: {float(test_acc):.4f}")

print("\n✓ K3-Node LabelPropagation execution completed successfully!")